In [1]:
from collections import defaultdict
import math

def affine_char_dict(cartan_type, level, max_ord=11):
    r'''
    Input:
      - cartan_type: Cartan type string (e.g., "A1", "A4") or CartanType object
      - level: positive integer (level k)
      - max_ord: maximum expansion order for q
    Output:
      - Dictionary mapping highest weight vectors to their non-specialised characters
    '''
    ct = CartanType(cartan_type)
    if ct.is_affine():
        ct = ct.classical()

    cartan_mat = ct.cartan_matrix()
    inv_cartan_mat = cartan_mat.inverse()

    d = vector(QQ, ct.symmetrizer())
    d /= max(d)

    G = diagonal_matrix(QQ, d) * cartan_mat
    quadF = inv_cartan_mat.transpose() * G * inv_cartan_mat

    ct_aff = ct.affine()
    index = list(ct_aff.index_set())
    WL = RootSystem(ct_aff).weight_lattice(extended=True)
    Lambda = WL.fundamental_weights()
    rnk = len(index) - 1

    # Simple coroots in fundamental weight coordinates
    C = cartan_mat * diagonal_matrix(QQ, [1/d[i] for i in range(rnk)])
    M = level * C
    inv_M = M.inverse()
    inv_C = C.inverse()

    A_mat = M.transpose() * quadF * M
    
    # Calculate Cholesky decomposition of A_mat via double precision floats for swift sphere traversal
    A_real = A_mat.change_ring(RDF)
    V_cholesky = A_real.cholesky().transpose()

    var_names = ['x%d' % (i + 1) for i in range(rnk)]
    Rx = LaurentPolynomialRing(ZZ, var_names)
    P = PuiseuxSeriesRing(Rx, 'q')
    q = P.gen()

    qF_gcd = 1 / gcd( quadF.list() )
    ram_ord = 2 * level * qF_gcd
    R2 = 2 * level * max_ord

    theta_cache = {}

    def get_points(u0, R2_val):
        """Finds all u in Z^rnk such that (u-u0)*A_mat*(u-u0) <= R2_val via Fincke-Pohst."""
        u0_real = [float(x) for x in u0]
        pts = []
        
        def search_sphere(k, current_R2, partial_sums, current_u):
            if k == -1:
                pts.append(tuple(current_u))
                return
                
            Vkk = V_cholesky[k, k]
            inv_Vkk = 1.0 / Vkk
            S = partial_sums[k]
            # Safety epsilon allows buffer for minute float rounding discrepancies
            bound = math.sqrt(max(0.0, current_R2)) + 1e-9
            
            min_y_k = (-bound - S) * inv_Vkk
            max_y_k = (bound - S) * inv_Vkk
            
            min_u_k = math.ceil(min_y_k + u0_real[k])
            max_u_k = math.floor(max_y_k + u0_real[k])
            
            for uk in range(int(min_u_k), int(max_u_k) + 1):
                yk = uk - u0_real[k]
                term = Vkk * yk + S
                new_R2 = current_R2 - term * term
                
                next_partial = list(partial_sums)
                for i in range(k):
                    next_partial[i] += V_cholesky[i, k] * yk
                
                # FIX: Explicitly assign coordinate uk for this branch
                next_u = list(current_u)
                next_u[k] = uk
                
                search_sphere(k - 1, new_R2, next_partial, next_u)
                
        search_sphere(rnk - 1, float(R2_val), [0.0] * rnk, [0] * rnk)
        
        # Exact filtering using rationals guarantees 100% mathematical precision
        u0_exact = vector(QQ, u0)
        valid_pts = []
        for pt in pts:
            u = vector(QQ, pt)
            du = u - u0_exact
            if du * A_mat * du <= R2_val:
                valid_pts.append(u)
        return valid_pts

    def Theta(v_lamb):
        v_key = tuple(v_lamb)
        if v_key in theta_cache:
            return theta_cache[v_key]

        u0 = - (1 / level) * inv_C * v_lamb
        
        # Native python dictionary tracks occurrences rapidly without object instantiation
        dictseries = defaultdict(lambda: defaultdict(int))
        
        valid_pts = get_points(u0, R2)
        
        for u in valid_pts:
            vec = M * u + v_lamb
            exponent = int(qF_gcd * (vec * quadF * vec))
            mono_key = int(vec[0]) if rnk == 1 else tuple(int(v) for v in vec)
            dictseries[exponent][mono_key] += 1

        # Generates polynomial ring elements purely as a batch operation at loop's end
        poly_dict = {exp: Rx(dict(monos)) for exp, monos in dictseries.items()}
        res = P(poly_dict, e=ram_ord).add_bigoh(max_ord)
        
        theta_cache[v_key] = res
        return res

    def get_classical_orbit(v_start):
        '''Generates the exact classical Weyl group orbit of a weight vector modulo kM'''
        v_start = vector(QQ, v_start)
        orbit = [v_start]
        queue = [v_start]
        
        # Extracts vector representation's fractional remainder to function as a hashed state
        def to_canon(v):
            coords = inv_M * v
            return tuple((c - floor(c)) for c in coords)
            
        visited = {to_canon(v_start)}
        
        while queue:
            curr = queue.pop(0)
            
            for i in range(rnk):
                nxt = curr - curr[i] * cartan_mat.column(i)
                canon_nxt = to_canon(nxt)
                
                # Modulo uniqueness checked in O(1)
                if canon_nxt not in visited:
                    visited.add(canon_nxt)
                    orbit.append(nxt)
                    queue.append(nxt)
        return orbit

    # Integrable weights setup
    Q_dual = RootSystem(ct_aff.dual()).root_lattice()
    comarks = {i: Q_dual.null_root().coefficient(i) for i in index}

    int_weights_coeffs = []
    def find_weights(idx, current_sum, current_dict):
        if idx == rnk + 1:
            if current_sum == level:
                int_weights_coeffs.append(current_dict.copy())
            return
        i = index[idx]
        max_val = (level - current_sum) // comarks[i]
        for val in range(max_val + 1):
            current_dict[i] = val
            find_weights(idx + 1, current_sum + val * comarks[i], current_dict)
            del current_dict[i]

    find_weights(0, 0, {})
    int_weights = [sum(Lambda[i] * int_weights_coeffs[j][i] for i in index) for j in range(len(int_weights_coeffs))]

    # Character Assembly
    char_dict = {}
    for iwght in int_weights:
        int_rep = IntegrableRepresentation(iwght)
        md_weights = int_rep.dominant_maximal_weights()

        char = P(0).add_bigoh(max_ord)
        for wght in md_weights:
            v_dom = wght.to_classical().to_vector()
            mod_char = int_rep.modular_characteristic(wght)
            str_func = int_rep.string(wght, depth=max_ord)
            
            theta_orbit_sum = P(0).add_bigoh(max_ord)
            for v_orb in get_classical_orbit(v_dom):
                theta_orbit_sum += Theta(v_orb)

            char += (q**mod_char) * P(str_func) * theta_orbit_sum

        char_dict[tuple( iwght.to_vector()[:-1] )] = char
        #char_dict[iwght] = char

    return char_dict

In [2]:
# Load a list of cosets
easy_cosets = load('easy_cosets')

In [3]:
def compute_coset_characters(num_chars, den_chars_list, weight_combos, P_list, max_ord=None):
    r"""
    Computes the branching functions (coset characters) for WZW coset models 
    of the form L_0 / (L_1 \times \dots \times L_n).

    Parameters
    ----------
    num_chars : dict
        Keys are numerator highest weights. Values are Puiseux series in `q` 
        with Laurent polynomial coefficients in the numerator fugacities.
    den_chars_list : list of dicts
        List of dictionaries for each denominator Lie algebra.
    weight_combos : list of tuples of tuples
        Valid combinations of weights: (wt_0, wt_1, ..., wt_n).
    P_list : list of matrices
        Projection matrices P_1, ..., P_n from the weight lattice of L_0 
        to the weight lattices of L_1, ..., L_n.
    max_ord : int or rational, optional
        The maximum order in `q` up to which the characters are computed.
    """
    from sage.all import vector, Matrix, QQ, ZZ, PuiseuxSeriesRing, Infinity, Integer
    from collections import defaultdict
    import copy

    # -------------------------------------------------------------------------
    # 1. Validation and Precision Check
    # -------------------------------------------------------------------------
    input_precisions = []
    for s in num_chars.values():
        if hasattr(s, 'prec') and s.prec() != Infinity:
            input_precisions.append(s.prec())
    for den_dict in den_chars_list:
        for s in den_dict.values():
            if hasattr(s, 'prec') and s.prec() != Infinity:
                input_precisions.append(s.prec())

    min_prec = min(input_precisions) if input_precisions else Infinity

    if max_ord is None:
        if min_prec != Infinity:
            max_ord = min_prec - 1
        else:
            max_ord = 10
    elif min_prec != Infinity and max_ord >= min_prec:
        raise ValueError(
            f"max_ord ({max_ord}) must be strictly less than the minimum "
            f"precision of the input series ({min_prec})."
        )

    # -------------------------------------------------------------------------
    # 2. Setup Global Projection Topology
    # -------------------------------------------------------------------------
    P_tot_rows = []
    r_i_list = []
    
    # Vertically stack all projection matrices into one master projection
    for P_mat in P_list:
        P = Matrix(ZZ, P_mat)
        r_i = P.nrows()
        r_i_list.append(r_i)
        for r in range(r_i):
            P_tot_rows.append(P.row(r))
            
    P_tot = Matrix(ZZ, P_tot_rows)
    R_tot = sum(r_i_list)
    num_rank = P_tot.ncols()

    # -------------------------------------------------------------------------
    # 3. Robust Conversion Utilities
    # -------------------------------------------------------------------------
    def series_to_dict(s, expected_rank):
        """Converts a Puiseux series to a nested dict with exact tuple lengths."""
        res = {}
        if hasattr(s, 'dict'):
            s_items = list(s.dict().items())
        else:
            s_items = list(zip(s.exponents(), s.coefficients()))

        for q_deg, coeff in s_items:
            res[q_deg] = {}
            if hasattr(coeff, 'dict'):
                for mono, val in coeff.dict().items():
                    if isinstance(mono, (int, Integer)) or not hasattr(mono, '__len__'):
                        mono_tuple = (int(mono),)
                    else:
                        mono_tuple = tuple(int(x) for x in mono)
                    
                    # Force tuple to the mathematically expected rank (0 padding)
                    if len(mono_tuple) < expected_rank:
                        mono_tuple += (0,) * (expected_rank - len(mono_tuple))
                    elif len(mono_tuple) > expected_rank:
                        mono_tuple = mono_tuple[:expected_rank]
                        
                    res[q_deg][mono_tuple] = ZZ(val)
            else:
                mono_tuple = (0,) * expected_rank
                res[q_deg][mono_tuple] = ZZ(coeff)
        return res

    def mult_series_dicts(s1, s2):
        """Multiplies two series represented as nested dicts."""
        res = {}
        for dq1, c1 in s1.items():
            for dq2, c2 in s2.items():
                dq = dq1 + dq2
                if dq > max_ord:
                    continue
                if dq not in res:
                    res[dq] = {}
                for m1, v1 in c1.items():
                    for m2, v2 in c2.items():
                        m = tuple(a + b for a, b in zip(m1, m2))
                        res[dq][m] = res[dq].get(m, 0) + v1 * v2
                        if res[dq][m] == 0:
                            del res[dq][m]
                if not res[dq]:
                    del res[dq]
        return res

    # Convert numerators and project them DOWN into the denominator space
    num_dicts_mapped = {}
    for wt, s in num_chars.items():
        s_dict = series_to_dict(s, num_rank)
        res = {}
        for dq, coeffs in s_dict.items():
            res[dq] = {}
            for m, v in coeffs.items():
                mu = vector(ZZ, m)
                new_m = tuple(P_tot * mu)
                res[dq][new_m] = res[dq].get(new_m, 0) + v
            if not res[dq]:
                del res[dq]
        if res:
            num_dicts_mapped[wt] = res

    # Convert denominators and embed them into the joint R_tot dimensional space
    den_dicts_mapped = []
    offset = 0
    for i, den_dict in enumerate(den_chars_list):
        r_i = r_i_list[i]
        mapped_dict = {}
        for wt, s in den_dict.items():
            s_dict = series_to_dict(s, r_i)
            res = {}
            for dq, coeffs in s_dict.items():
                res[dq] = {}
                for m, v in coeffs.items():
                    new_m = (0,) * offset + m + (0,) * (R_tot - offset - r_i)
                    res[dq][new_m] = res[dq].get(new_m, 0) + v
                if not res[dq]:
                    del res[dq]
            if res:
                mapped_dict[wt] = res
        den_dicts_mapped.append(mapped_dict)
        offset += r_i

    # -------------------------------------------------------------------------
    # 4. Solve Linear System Order-by-Order for Each Numerator
    # -------------------------------------------------------------------------
    combos_by_num = defaultdict(list)
    for combo in weight_combos:
        combos_by_num[combo[0]].append(combo)

    coset_chars_dict = {}

    for wt_0, combos in combos_by_num.items():
        N_a = len(combos)
        
        # Precompute character products for denominator combinations
        D_a_list = []
        for combo in combos:
            D_a = {0: {tuple([0] * R_tot): 1}}
            for i in range(1, len(combo)):
                wt_i = combo[i]
                D_a = mult_series_dicts(D_a, den_dicts_mapped[i-1][wt_i])
            D_a_list.append(D_a)

        L_a_list = [min(D_a.keys()) if D_a else Infinity for D_a in D_a_list]
        R = copy.deepcopy(num_dicts_mapped.get(wt_0, {}))
        C_res = [{} for _ in range(N_a)]

        # Iterative Forward Substitution
        while R:
            K = min(R.keys())
            if K > max_ord:
                break
            if not R[K]:
                del R[K]
                continue
                
            M_set = set(R[K].keys())
            for a in range(N_a):
                L_a = L_a_list[a]
                if L_a != Infinity and L_a in D_a_list[a]:
                    M_set.update(D_a_list[a][L_a].keys())
            
            M_list = list(M_set)
            matrix_rows, matrix_cols = len(M_list), N_a
            
            A = Matrix(QQ, matrix_rows, matrix_cols)
            B = vector(QQ, matrix_rows)
            
            for i_row, m in enumerate(M_list):
                B[i_row] = R[K].get(m, 0)
                for a in range(N_a):
                    L_a = L_a_list[a]
                    if L_a != Infinity:
                        A[i_row, a] = D_a_list[a][L_a].get(m, 0)
            
            try:
                X = A.solve_right(B)
            except ValueError:
                raise RuntimeError(
                    f"Inconsistent linear system at q-degree {K} for numerator {wt_0}. "
                    f"Verify selection rules or precision bounds."
                )
                
            for a in range(N_a):
                try:
                    c = ZZ(X[a])
                except TypeError:
                    raise RuntimeError(f"Non-integer coefficient {X[a]} detected in coset character.")
                
                if c != 0:
                    L_a = L_a_list[a]
                    deg = K - L_a
                    C_res[a][deg] = C_res[a].get(deg, 0) + c
                    
                    for dq, D_coeff in D_a_list[a].items():
                        new_dq = dq + deg
                        if new_dq > max_ord:
                            continue
                        if new_dq not in R:
                            R[new_dq] = {}
                        for m, v in D_coeff.items():
                            R[new_dq][m] = R[new_dq].get(m, 0) - c * v
                            if R[new_dq][m] == 0:
                                del R[new_dq][m]
                        if not R[new_dq]:
                            del R[new_dq]
                            
            if K in R:
                del R[K]

        # ---------------------------------------------------------------------
        # 5. Puiseux Series Reconstruction
        # ---------------------------------------------------------------------
        R_q = PuiseuxSeriesRing(ZZ, 'q')
        q = R_q.gen()
        
        for a, combo in enumerate(combos):
            series_val = sum(c * q**deg for deg, c in C_res[a].items())
            if not isinstance(series_val, type(q)):
                series_val = R_q(series_val)
            coset_chars_dict[combo] = series_val

    return coset_chars_dict

In [4]:
max_ord = 10
coset = easy_cosets[1]
num_chars = affine_char_dict(*coset[1], max_ord)
den_chars_list = [ affine_char_dict(cartan_type, level, max_ord) for cartan_type,level in coset[2] ]
weight_combos = coset[5]
P_list = [ P for _,P in coset[3] ]

In [6]:
coset_char = compute_coset_characters(num_chars, den_chars_list, weight_combos, P_list, max_ord=max_ord-2)
coset_char

{((0, 0, 0, 0, 1, 0),
  (0, 3),
  (1,
   2)): q^(5/12) + 2*q^(17/12) + 4*q^(29/12) + 8*q^(41/12) + 14*q^(53/12) + 24*q^(65/12) + 40*q^(77/12),
 ((0, 0, 0, 0, 1, 0),
  (0, 3),
  (3,
   0)): q^(49/60) + 2*q^(109/60) + 3*q^(169/60) + 6*q^(229/60) + 11*q^(289/60) + 18*q^(349/60) + 30*q^(409/60),
 ((0, 0, 0, 0, 1, 0),
  (1, 2),
  (0,
   3)): q^(5/12) + 2*q^(17/12) + 4*q^(29/12) + 8*q^(41/12) + 14*q^(53/12) + 24*q^(65/12) + 40*q^(77/12),
 ((0, 0, 0, 0, 1, 0),
  (1, 2),
  (2,
   1)): q^(1/60) + 2*q^(61/60) + 5*q^(121/60) + 10*q^(181/60) + 18*q^(241/60) + 32*q^(301/60) + 53*q^(361/60) + 86*q^(421/60),
 ((0, 0, 0, 0, 1, 0),
  (2, 1),
  (1,
   2)): q^(1/60) + 2*q^(61/60) + 5*q^(121/60) + 10*q^(181/60) + 18*q^(241/60) + 32*q^(301/60) + 53*q^(361/60) + 86*q^(421/60),
 ((0, 0, 0, 0, 1, 0),
  (2, 1),
  (3,
   0)): q^(5/12) + 2*q^(17/12) + 4*q^(29/12) + 8*q^(41/12) + 14*q^(53/12) + 24*q^(65/12) + 40*q^(77/12) + 64*q^(89/12),
 ((0, 0, 0, 0, 1, 0),
  (3, 0),
  (0,
   3)): q^(49/60) + 2*q^(109/60) + 3*q

In [7]:
len(coset_char)

16

In [8]:
from collections import defaultdict
def affine_char_list(cartan_type, level, max_ord=15):
    '''
    Input: a cartan type (e.g. "A4"), a level, and the maximum order to compute the characters to
    Output: the list of normalised specialised characters of the affine lie algebra
    '''
    # Get the Cartan type
    ct = CartanType(cartan_type)

    # Make sure given Cartan type is affine
    if ct.is_affine():
        ct = ct.classical()

    # Cartan Matrix
    cartan_mat = ct.cartan_matrix()
    inv_cartan_mat = cartan_mat.inverse()

    # After normalization, d_i = (alpha_i, alpha_i)/2.
    d = vector(QQ, ct.symmetrizer())
    d /= max(d)

    # Gram matrix of simple roots.
    G = diagonal_matrix(QQ, d) * cartan_mat

    if G != G.transpose():
        raise ValueError("DA is not symmetric")

    # Gram matrix of fundamental weights.
    quadF = inv_cartan_mat.transpose() * G * inv_cartan_mat
    max_F = max(quadF.coefficients())

    # Root system.
    R = ct.root_system().root_lattice()
    # Highest root (multiply by A to convert to fundamental weight coordinates).
    highest_root = cartan_mat * vector(ZZ, R.highest_root())

    # Convert to affine Cartan type
    ct = ct.affine()

    # Indices
    index = list( ct.index_set() )
    # Affine root system
    WL = RootSystem(ct).weight_lattice(extended=True)
    # Simple roots
    simple_roots = WL.simple_roots()
    # Simple coroots
    simple_coroots = WL.simple_coroots()
    # List of fundamental weights
    Lambda = WL.fundamental_weights()
    # The rank
    rnk = len(index)-1
    # The null root delta
    delta = WL.null_root()
    
    # Now we find the (classical) weight, root, and coroot lattices as sagemath free module objects over the integers
    weight_lattice = span([vector(ZZ,rnk,{i:1}) for i in range(rnk)],ZZ)
    root_lattice = span(cartan_mat.columns(),ZZ)
    coroot_lattice = span([cartan_mat.columns()[i]*highest_root.norm_squared()/simple_roots[i+1].norm_squared() for i in range(rnk)],ZZ)

    # The power series ring 
    P = PuiseuxSeriesRing(ZZ,'q')
    q = P.gen()

    # Define the generalised theta functions
    def Theta(lamb):
        r'''
        Generalised theta function
        Input: a weight
        Output: the specialised generalised theta function associated to that weight
        '''
        vec_lamb = lamb.to_classical().to_vector()
        dictseries = defaultdict(int)
        qF_gcd = 1/gcd(quadF.coefficients())
        e = 2*level*qF_gcd
        for cr in coroot_lattice:
            vec = level*cr+vec_lamb
            exponent = qF_gcd*vec*quadF*vec
            if exponent <= e*max_ord:
                dictseries[exponent] += 1
            elif exponent > 2*e*max_F*max_ord:
                break
        
        return P(dictseries,e=e).add_bigoh(max_ord)
  

    # Get the affine Weyl group and isolate the classical generators
    affine_weyl = WL.weyl_group()
    classical_nodes = [i for i in index if i != 0]
    weyl_gen = [affine_weyl.simple_reflection(i) for i in classical_nodes]

    # Comarks of ct_affine equal the marks of its dual affine type
    # We find them by looking at the null root (delta) of the dual root lattice
    Q_dual = RootSystem(ct.dual()).root_lattice()
    delta_dual = Q_dual.null_root()
    comarks = {i: delta_dual.coefficient(i) for i in index}

    # Now we find all the integrable weights
    int_weights_coeffs = []
    def find_weights(idx, current_sum, current_dict):
        if idx == rnk+1:
            if current_sum == level:
                int_weights_coeffs.append(current_dict.copy())
            return
        i = index[idx]
        max_val = (level - current_sum) // comarks[i] 
        for val in range(max_val + 1):
            current_dict[i] = val
            find_weights(idx + 1, current_sum + val * comarks[i], current_dict)
            del current_dict[i]
            
    find_weights(0, 0, {})
    # Number of integrable highest weight representations
    num_reps = len(int_weights_coeffs)
    # The integrable weights
    int_weights = [ sum(Lambda[i]*int_weights_coeffs[j][i] for i in index) for j in range(num_reps)] 
    def are_equivalent(w1, w2):
        '''Function to determine if two weights are related via shift by a coroot'''

        c1 = vector(QQ, [w1.monomial_coefficients().get(i+1,0) for i in range(rnk)] )
        c2 = vector(QQ, [w2.monomial_coefficients().get(i+1,0) for i in range(rnk)] )
        return (c1-c2)/level in coroot_lattice #and w1.monomial_coefficients().get(0,0)==w2.monomial_coefficients().get(0,0)

    # Now loop over these weights and calculate the character for each
    char_list = []
    for iwght in int_weights:
        # The corresponding integrable representation
        int_rep = IntegrableRepresentation(iwght)
        # First let's calculate the set of maximal dominant weights
        md_weights = int_rep.dominant_maximal_weights()
        # Now we find a full set of representatives up to translations by the coroot lattice
        full_maximal_representatives = list( md_weights )
        queue = list( md_weights )
        while queue:
            current = queue.pop(0)
            for g in weyl_gen:
                nxt = g.action(current)
                # Only keep nxt if it represents a brand new class modulo the coroots
                if not any(are_equivalent(nxt, r) for r in full_maximal_representatives):
                    full_maximal_representatives.append(nxt)
                    queue.append(nxt)
        
        # Now compute the character by summing over these weights
        char = P(0).add_bigoh(max_ord)
        for wght in full_maximal_representatives:
            char += q**(int_rep.modular_characteristic(wght))*P( int_rep.string(wght, depth=max_ord) )*Theta(wght)
        # Update the list of characters
        char_list.append(char)
        
    return char_list